# Hybrid RAG with LangGraph Supervisor Agent

> **Neo4j GraphRAG + Elasticsearch VectorDB를 결합한 하이브리드 검색 시스템**


## 1. Hybrid RAG란?

### 1.1 단일 RAG의 한계

**GraphRAG (Neo4j)의 강점과 한계**:
- ✅ **강점**: 엔티티 간 관계 파악, 구조화된 쿼리, 정확한 관계 추론
- ❌ **한계**: 의미적 유사도 검색 약함, 자연어 검색 제한적

**VectorRAG (Elasticsearch)의 강점과 한계**:
- ✅ **강점**: 의미적 유사도 검색 우수, 자연어 검색 강력, 키워드 검색(BM25) 지원
- ❌ **한계**: 관계 정보 손실, 복잡한 관계 추론 어려움

### 1.2 Hybrid RAG의 필요성

```
사용자 질문: "경제 뉴스 중에서 네이버와 관련된 기사를 찾아줘"

┌─────────────────────────────────────────────────┐
│         Supervisor Agent (조율자)                │
│   - 질문 분석                                     │
│   - 적절한 Agent 선택 및 실행                      │
│   - 결과 통합 및 최종 답변 생성                     │
└─────────────────────────────────────────────────┘
            │                    │
            ↓                    ↓
┌──────────────────┐    ┌──────────────────┐
│  GraphRAG Agent  │    │ VectorRAG Agent  │
│    (Neo4j)       │    │ (Elasticsearch)  │
│                  │    │                  │
│ - 구조적 검색     │    │ - 의미적 검색     │
│ - 관계 기반 쿼리  │    │ - 키워드 검색     │
│ - 카테고리/언론사 │    │ - 유사도 검색     │
└──────────────────┘    └──────────────────┘
```

**Hybrid RAG의 장점**:
- ✅ 각 검색 시스템의 장점 활용
- ✅ 다양한 질문 유형에 대응
- ✅ 검색 정확도 및 재현율(Recall) 향상
- ✅ 중복 제거 및 결과 통합


## 2. LangGraph란?

### 2.1 LangGraph 개요

> **LangGraph**는 LangChain 팀이 개발한 **상태 기반 멀티 에이전트 오케스트레이션 프레임워크**입니다.

**주요 특징**:
- **그래프 기반 워크플로우**: 노드(Agent)와 엣지(전환)로 구성
- **상태 관리**: 전역 상태를 통한 에이전트 간 정보 공유
- **유연한 제어 흐름**: 조건부 라우팅, 루프, 병렬 실행
- **Human-in-the-Loop**: 사람의 개입 지점 설정 가능

### 2.2 Supervisor Agent 패턴

```
           ┌─────────────────┐
           │   Supervisor    │  ← 전체 워크플로우 조율
           │     Agent       │
           └────────┬────────┘
                    │
        ┌───────────┴───────────┐
        │                       │
        ↓                       ↓
┌───────────────┐       ┌───────────────┐
│  Worker       │       │  Worker       │
│  Agent 1      │       │  Agent 2      │
│ (GraphRAG)    │       │ (VectorRAG)   │
└───────────────┘       └───────────────┘
```

**Supervisor의 역할**:
1. 사용자 질문 분석
2. 적절한 Worker Agent 선택
3. Agent 실행 및 결과 수집
4. 필요시 추가 Agent 호출
5. 최종 답변 생성


## 3. 환경 설정 및 초기화


### 3.1 필요한 라이브러리 임포트


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv

# 환경변수 로드
load_dotenv()


### 3.2 Neo4j 연결 (GraphRAG)


In [ ]:
from neo4j import GraphDatabase

class Singleton(type):
    _instances = {}
    
    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super(Singleton, cls).__call__(*args, **kwargs)
        return cls._instances[cls]

class Neo4jConnection(metaclass=Singleton):
    """Neo4j 데이터베이스 연결 및 작업을 위한 클래스"""
    
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        print("✅ Neo4j 연결 성공!")
    
    def close(self):
        if self.driver is not None:
            self.driver.close()
            print("Neo4j 연결 종료")
    
    def execute_query(self, query, parameters=None):
        with self.driver.session() as session:
            result = session.run(query, parameters)
            return [record for record in result]

# Neo4j 연결
neo4j_conn = Neo4jConnection(
    uri="bolt://localhost:7687",
    user="neo4j",
    password="test1234"
)


### 3.3 Elasticsearch 연결 (VectorRAG)


In [ ]:
from elasticsearch import Elasticsearch

# Elasticsearch 클라이언트 생성
es_client = Elasticsearch(
    ["http://localhost:9200"],
    basic_auth=("elastic", "changeme123!"),
    verify_certs=False,
    ssl_show_warn=False,
    request_timeout=30,
    max_retries=3,
    retry_on_timeout=True,
    headers={"accept": "application/json", "content-type": "application/json"}
)

if es_client.ping():
    print("✅ Elasticsearch 연결 성공!")
    info = es_client.info()
    print(f"   버전: {info['version']['number']}")
else:
    print("❌ Elasticsearch 연결 실패")


### 3.4 임베딩 모델 및 LLM 설정


In [ ]:
from langchain_ollama import OllamaEmbeddings
from langchain_openai import ChatOpenAI

# 임베딩 모델
embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b")

# LLM 모델
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

print("✅ 임베딩 모델 및 LLM 초기화 완료")


## 4. GraphRAG Agent 구현 (Neo4j)


### 4.1 Cypher 쿼리 템플릿

GraphRAG는 구조화된 Cypher 쿼리를 통해 관계 기반 검색을 수행합니다.


In [ ]:
from enum import Enum

class CypherQueryTemplates(Enum):
    """Cypher 쿼리 템플릿"""

    NEWS_BY_CATEGORY = "news_by_category"
    NEWS_BY_PUBLISHER = "news_by_publisher"
    NEWS_BY_REPORTER = "news_by_reporter"

    def build(self, **kwargs):
        """템플릿 유형에 따라 Cypher 쿼리 생성"""
        if self is CypherQueryTemplates.NEWS_BY_CATEGORY:
            category_name = kwargs["category_name"]
            limit_no = kwargs.get("limit_no", 3)
            return f"""
            MATCH (n:News)-[:BELONGS_TO]->(c:Category {{name: "{category_name}"}})
            MATCH (n)-[:PUBLISHED_BY]->(p:Publisher)
            MATCH (n)-[:WRITTEN_BY]->(r:Reporter)
            RETURN n.title as 제목, n.content as 내용, 
                   p.name as 언론사, r.name as 기자, n.publishDate as 발행일,
                   n.link as 뉴스링크
            LIMIT {limit_no}
            """

        if self is CypherQueryTemplates.NEWS_BY_PUBLISHER:
            publisher_name = kwargs["publisher_name"]
            limit_no = kwargs.get("limit_no", 3)
            return f"""
            MATCH (n:News)-[:PUBLISHED_BY]->(p:Publisher {{name: "{publisher_name}"}})
            MATCH (n)-[:BELONGS_TO]->(c:Category)
            MATCH (n)-[:WRITTEN_BY]->(r:Reporter)
            RETURN n.title as 제목, n.content as 내용, 
                   c.name as 카테고리, r.name as 기자, n.publishDate as 발행일,
                   n.link as 뉴스링크
            LIMIT {limit_no}
            """

        if self is CypherQueryTemplates.NEWS_BY_REPORTER:
            reporter_name = kwargs["reporter_name"]
            limit_no = kwargs.get("limit_no", 3)
            return f"""
            MATCH (n:News)-[:WRITTEN_BY]->(r:Reporter {{name: "{reporter_name}"}})
            MATCH (n)-[:BELONGS_TO]->(c:Category)
            MATCH (n)-[:PUBLISHED_BY]->(p:Publisher)
            RETURN n.title as 제목, n.content as 내용, 
                   c.name as 카테고리, p.name as 언론사, n.publishDate as 발행일,
                   n.link as 뉴스링크
            LIMIT {limit_no}
            """

        raise ValueError(f"지원하지 않는 템플릿: {self}")

print("✅ Cypher 쿼리 템플릿 정의 완료")


### 4.2 GraphRAG Agent 함수


In [ ]:
def graph_rag_search(query_type: str, **kwargs) -> str:
    """
    Neo4j GraphRAG 검색 함수
    
    Args:
        query_type: 쿼리 타입 (category, publisher, reporter)
        **kwargs: 검색 파라미터
    
    Returns:
        검색 결과 문자열
    """
    try:
        # 쿼리 타입에 따라 템플릿 선택
        if query_type == "category":
            template = CypherQueryTemplates.NEWS_BY_CATEGORY
            query = template.build(
                category_name=kwargs.get("category_name"),
                limit_no=kwargs.get("limit_no", 3)
            )
        elif query_type == "publisher":
            template = CypherQueryTemplates.NEWS_BY_PUBLISHER
            query = template.build(
                publisher_name=kwargs.get("publisher_name"),
                limit_no=kwargs.get("limit_no", 3)
            )
        elif query_type == "reporter":
            template = CypherQueryTemplates.NEWS_BY_REPORTER
            query = template.build(
                reporter_name=kwargs.get("reporter_name"),
                limit_no=kwargs.get("limit_no", 3)
            )
        else:
            return f"지원하지 않는 쿼리 타입: {query_type}"
        
        # Neo4j 쿼리 실행
        results = neo4j_conn.execute_query(query)
        
        if not results:
            return f"검색 결과가 없습니다. (쿼리 타입: {query_type})"
        
        # 결과 포맷팅
        output = f"[GraphRAG 검색 결과 - {query_type}]\n\n"
        for i, record in enumerate(results, 1):
            output += f"{i}. 제목: {record['제목']}\n"
            output += f"   내용: {record['내용'][:200]}...\n"
            if '카테고리' in record.keys():
                output += f"   카테고리: {record['카테고리']}\n"
            if '언론사' in record.keys():
                output += f"   언론사: {record['언론사']}\n"
            if '기자' in record.keys():
                output += f"   기자: {record['기자']}\n"
            output += f"   발행일: {record['발행일']}\n"
            output += f"   링크: {record['뉴스링크']}\n\n"
        
        return output
    
    except Exception as e:
        return f"GraphRAG 검색 중 오류 발생: {str(e)}"

# 테스트
print(graph_rag_search("category", category_name="경제", limit_no=2))


### 5.1 Elasticsearch VectorStore 클래스


In [ ]:
from langchain_core.vectorstores.base import VectorStore
from langchain_core.documents import Document
from typing import List

class ElasticsearchVectorStore(VectorStore):
    """Elasticsearch 기반 VectorStore"""

    def __init__(self, es_client, index_name, embeddings):
        self.es_client = es_client
        self.index_name = index_name
        self._embeddings = embeddings

    @classmethod
    def from_texts(cls, **kwargs):
        """VectorStore 상속을 위한 필수 함수"""
        pass

    def __search_hybrid(self, query: str, k: int):
        """하이브리드 검색 (벡터 + 키워드)"""
        query_embedding = self._embeddings.embed_query(query)
        
        search_query = {
            "query": {
                "bool": {
                    "should": [
                        {
                            "match": {
                                "text": {
                                    "query": query,
                                    "boost": 1.0
                                }
                            }
                        }
                    ]
                }
            },
            "knn": {
                "field": "embedding",
                "query_vector": query_embedding,
                "k": k,
                "num_candidates": 100,
                "boost": 2.0
            },
            "size": k,
            "_source": ["text", "metadata"]
        }
        
        return self.es_client.search(index=self.index_name, body=search_query)

    def similarity_search(self, query: str, k: int = 3) -> List[Document]:
        """하이브리드 검색 (벡터 + 키워드)"""
        response = self.__search_hybrid(query, k)
        
        documents = []
        for hit in response['hits']['hits']:
            doc = Document(
                page_content=hit['_source']['text'],
                metadata=hit['_source'].get('metadata', {})
            )
            documents.append(doc)
        
        return documents

# VectorStore 초기화
vectorstore = ElasticsearchVectorStore(
    es_client=es_client,
    index_name="naver_news",
    embeddings=embeddings
)

print("✅ Elasticsearch VectorStore 초기화 완료")


### 5.2 VectorRAG Agent 함수


In [ ]:
def vector_rag_search(query: str, k: int = 3) -> str:
    """
    Elasticsearch VectorRAG 검색 함수
    
    Args:
        query: 검색 쿼리
        k: 반환할 문서 수
    
    Returns:
        검색 결과 문자열
    """
    try:
        # 하이브리드 검색 (벡터 + 키워드)
        results = vectorstore.similarity_search(query, k=k)
        
        if not results:
            return "검색 결과가 없습니다."
        
        # 결과 포맷팅
        output = f"[VectorRAG 검색 결과 - 하이브리드]\n\n"
        for i, doc in enumerate(results, 1):
            output += f"{i}. {doc.page_content[:300]}...\n"
            output += f"   메타데이터:\n"
            for key, value in doc.metadata.items():
                output += f"     - {key}: {value}\n"
            output += "\n"
        
        return output
    
    except Exception as e:
        return f"VectorRAG 검색 중 오류 발생: {str(e)}"

# 테스트
print(vector_rag_search("네이버 관련 뉴스", k=2))


## 6. LangGraph Supervisor Agent 구현


### 6.1 상태 정의

LangGraph에서 사용할 전역 상태를 정의합니다.


In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    """Agent 간 공유되는 상태"""
    messages: Annotated[list, add_messages]  # 대화 메시지 목록
    next: str  # 다음 실행할 Agent
    graph_results: str  # GraphRAG 검색 결과
    vector_results: str  # VectorRAG 검색 결과
    final_answer: str  # 최종 답변

print("✅ Agent 상태 정의 완료")


### 6.2 Supervisor Agent 노드

Supervisor는 질문을 분석하고 적절한 Worker Agent를 선택합니다.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def supervisor_node(state: AgentState) -> AgentState:
    """
    Supervisor Agent: 질문을 분석하고 적절한 Agent 선택
    """
    messages = state["messages"]
    user_question = messages[-1].content
    
    # Supervisor 프롬프트
    supervisor_prompt = ChatPromptTemplate.from_messages([
        ("system", """당신은 하이브리드 RAG 시스템의 Supervisor입니다.
사용자 질문을 분석하고 적절한 검색 전략을 선택하세요.

사용 가능한 Agent:
1. **graph_agent**: 구조화된 검색에 적합 (카테고리, 언론사, 기자명 기반)
   - 예: "경제 카테고리 뉴스", "뉴스1에서 발행한 기사", "김철수 기자가 쓴 기사"
   
2. **vector_agent**: 의미적 검색에 적합 (자연어, 키워드 기반)
   - 예: "AI 관련 뉴스", "부산 박물관 소식", "해킹 사고 관련 기사"
   
3. **both**: 두 Agent를 모두 사용 (복합적인 질문)
   - 예: "경제 카테고리에서 네이버 관련 뉴스"

질문을 분석하고 다음 중 하나를 선택하세요: graph_agent, vector_agent, both
반드시 선택한 agent 이름만 출력하세요."""),
        ("human", "{question}")
    ])
    
    chain = supervisor_prompt | llm | StrOutputParser()
    decision = chain.invoke({"question": user_question}).strip().lower()
    
    # 결정 정규화
    if "both" in decision:
        next_agent = "both"
    elif "graph" in decision:
        next_agent = "graph_agent"
    elif "vector" in decision:
        next_agent = "vector_agent"
    else:
        next_agent = "both"  # 기본값
    
    print(f"🎯 Supervisor 결정: {next_agent}")
    
    state["next"] = next_agent
    return state

print("✅ Supervisor 노드 정의 완료")


### 6.3 Worker Agent 노드들


In [ ]:
def graph_agent_node(state: AgentState) -> AgentState:
    """GraphRAG Agent: Neo4j 기반 구조적 검색"""
    messages = state["messages"]
    user_question = messages[-1].content
    
    print("📊 GraphRAG Agent 실행 중...")
    
    # 질문에서 파라미터 추출
    extraction_prompt = ChatPromptTemplate.from_messages([
        ("system", """질문을 분석하고 Neo4j 검색에 필요한 정보를 추출하세요.

카테고리 목록: 경제, 생활/문화, IT/과학, 세계

다음 형식으로 응답하세요:
query_type: category 또는 publisher 또는 reporter
parameter: 추출된 값

예시:
- "경제 뉴스" → query_type: category, parameter: 경제
- "뉴스1 기사" → query_type: publisher, parameter: 뉴스1
- "김철수 기자" → query_type: reporter, parameter: 김철수 기자"""),
        ("human", "{question}")
    ])
    
    chain = extraction_prompt | llm | StrOutputParser()
    extracted = chain.invoke({"question": user_question})
    
    # 파라미터 파싱
    query_type = None
    parameter = None
    
    for line in extracted.split("\n"):
        if "query_type" in line.lower():
            query_type = line.split(":")[-1].strip()
        elif "parameter" in line.lower():
            parameter = line.split(":")[-1].strip()
    
    # GraphRAG 검색 실행
    if query_type and parameter:
        if query_type == "category":
            results = graph_rag_search("category", category_name=parameter, limit_no=3)
        elif query_type == "publisher":
            results = graph_rag_search("publisher", publisher_name=parameter, limit_no=3)
        elif query_type == "reporter":
            results = graph_rag_search("reporter", reporter_name=parameter, limit_no=3)
        else:
            results = "파라미터 추출 실패"
    else:
        # 파라미터 추출 실패 시 기본 검색
        results = "GraphRAG 검색을 위한 구조화된 정보를 찾을 수 없습니다."
    
    state["graph_results"] = results
    print("✅ GraphRAG 검색 완료")
    return state


def vector_agent_node(state: AgentState) -> AgentState:
    """VectorRAG Agent: Elasticsearch 기반 의미적 검색"""
    messages = state["messages"]
    user_question = messages[-1].content
    
    print("🔍 VectorRAG Agent 실행 중...")
    
    # VectorRAG 검색 실행
    results = vector_rag_search(user_question, k=3)
    
    state["vector_results"] = results
    print("✅ VectorRAG 검색 완료")
    return state

print("✅ Worker Agent 노드 정의 완료")


In [ ]:
def generate_answer_node(state: AgentState) -> AgentState:
    """검색 결과를 종합하여 최종 답변 생성"""
    messages = state["messages"]
    user_question = messages[-1].content
    graph_results = state.get("graph_results", "")
    vector_results = state.get("vector_results", "")
    
    print("📝 최종 답변 생성 중...")
    
    # 답변 생성 프롬프트
    answer_prompt = ChatPromptTemplate.from_messages([
        ("system", """당신은 뉴스 검색 시스템의 AI 어시스턴트입니다.
검색 결과를 바탕으로 사용자 질문에 대한 명확하고 유용한 답변을 생성하세요.

답변 작성 가이드:
1. 검색된 뉴스 기사들을 요약하여 제시
2. 각 기사의 제목, 주요 내용, 출처를 포함
3. 사용자가 더 자세히 알 수 있도록 링크 제공
4. 검색 결과가 없으면 정중하게 안내

한국어로 답변하세요."""),
        ("human", """질문: {question}

GraphRAG 검색 결과:
{graph_results}

VectorRAG 검색 결과:
{vector_results}

위 검색 결과를 종합하여 답변을 생성하세요.""")
    ])
    
    chain = answer_prompt | llm | StrOutputParser()
    answer = chain.invoke({
        "question": user_question,
        "graph_results": graph_results if graph_results else "검색 결과 없음",
        "vector_results": vector_results if vector_results else "검색 결과 없음"
    })
    
    state["final_answer"] = answer
    print("✅ 최종 답변 생성 완료")
    return state

print("✅ 답변 생성 노드 정의 완료")


In [ ]:
def route_after_supervisor(state: AgentState) -> str:
    """Supervisor의 결정에 따라 라우팅"""
    next_agent = state.get("next", "both")
    
    if next_agent == "graph_agent":
        return "graph_agent"
    elif next_agent == "vector_agent":
        return "vector_agent"
    else:  # both
        return "both"

def route_after_graph(state: AgentState) -> str:
    """GraphRAG 실행 후 라우팅"""
    next_agent = state.get("next", "both")
    
    if next_agent == "both":
        return "vector_agent"
    else:
        return "generate_answer"

def route_after_vector(state: AgentState) -> str:
    """VectorRAG 실행 후 라우팅"""
    return "generate_answer"

print("✅ 라우팅 함수 정의 완료")


### 6.6 LangGraph 구성


In [ ]:
from langgraph.graph import StateGraph, END

# 그래프 생성
workflow = StateGraph(AgentState)

# 노드 추가
workflow.add_node("supervisor", supervisor_node)
workflow.add_node("graph_agent", graph_agent_node)
workflow.add_node("vector_agent", vector_agent_node)
workflow.add_node("generate_answer", generate_answer_node)

# 시작점 설정
workflow.set_entry_point("supervisor")

# 엣지 추가 (Supervisor → Worker Agents)
workflow.add_conditional_edges(
    "supervisor",
    route_after_supervisor,
    {
        "graph_agent": "graph_agent",
        "vector_agent": "vector_agent",
        "both": "graph_agent"  # both인 경우 graph_agent 먼저 실행
    }
)

# GraphRAG Agent 이후 라우팅
workflow.add_conditional_edges(
    "graph_agent",
    route_after_graph,
    {
        "vector_agent": "vector_agent",
        "generate_answer": "generate_answer"
    }
)

# VectorRAG Agent 이후 → 답변 생성
workflow.add_edge("vector_agent", "generate_answer")

# 답변 생성 후 종료
workflow.add_edge("generate_answer", END)

# 그래프 컴파일
app = workflow.compile()

print("✅ LangGraph 워크플로우 구성 완료")


### 6.7 워크플로우 시각화


In [ ]:
from IPython.display import Image, display

try:
    # 그래프 시각화
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"그래프 시각화 실패: {e}")
    print("워크플로우 구조:")
    print("""
    START
      ↓
    Supervisor (질문 분석 및 Agent 선택)
      ↓
    ┌─────────┬─────────┐
    │         │         │
    ↓         ↓         ↓
  Graph    Vector    Both
  Agent    Agent   (Graph → Vector)
    │         │         │
    └─────────┴─────────┘
              ↓
        Generate Answer
              ↓
            END
    """)


## 7. Hybrid RAG 시스템 테스트


### 7.1 헬퍼 함수


In [ ]:
def run_hybrid_rag(question: str, verbose: bool = True):
    """
    Hybrid RAG 시스템 실행
    
    Args:
        question: 사용자 질문
        verbose: 상세 출력 여부
    """
    print("=" * 80)
    print(f"질문: {question}")
    print("=" * 80)
    
    # 초기 상태
    initial_state = {
        "messages": [{"role": "user", "content": question}],
        "next": "",
        "graph_results": "",
        "vector_results": "",
        "final_answer": ""
    }
    
    # 워크플로우 실행
    result = app.invoke(initial_state)
    
    if verbose:
        print("\n" + "=" * 80)
        print("실행 과정")
        print("=" * 80)
        
        if result.get("graph_results"):
            print("\n📊 GraphRAG 검색 결과:")
            print("-" * 80)
            print(result["graph_results"][:500])
            if len(result["graph_results"]) > 500:
                print(f"... (총 {len(result['graph_results'])}자)")
        
        if result.get("vector_results"):
            print("\n🔍 VectorRAG 검색 결과:")
            print("-" * 80)
            print(result["vector_results"][:500])
            if len(result["vector_results"]) > 500:
                print(f"... (총 {len(result['vector_results'])}자)")
    
    print("\n" + "=" * 80)
    print("최종 답변")
    print("=" * 80)
    print(result["final_answer"])
    print("=" * 80 + "\n")
    
    return result

print("✅ 헬퍼 함수 정의 완료")


### 7.2 테스트 1: 구조적 검색 (GraphRAG 선호)

카테고리 기반 검색은 GraphRAG가 적합합니다.


In [ ]:
result = run_hybrid_rag("경제 카테고리의 최신 뉴스를 찾아줘")


### 7.3 테스트 2: 의미적 검색 (VectorRAG 선호)

키워드 및 의미적 검색은 VectorRAG가 적합합니다.


In [ ]:
result = run_hybrid_rag("AI와 인공지능 관련 뉴스 보여줘")


### 7.4 테스트 3: 하이브리드 검색 (Both)

구조적 + 의미적 검색이 모두 필요한 복합 질문입니다.


In [ ]:
result = run_hybrid_rag("IT/과학 카테고리에서 네이버와 관련된 뉴스를 찾아줘")


### 7.5 테스트 4: 언론사 기반 검색


In [ ]:
result = run_hybrid_rag("뉴스1에서 발행한 기사들을 보여줘")


### 7.6 테스트 5: 복잡한 질문


In [ ]:
result = run_hybrid_rag("부산이나 박물관과 관련된 문화 뉴스가 있어?")


## 8. Hybrid RAG의 장점 분석


### 8.1 단일 RAG vs Hybrid RAG 비교

| 측면 | 단일 RAG | Hybrid RAG (우리 시스템) |
|------|---------|------------------------|
| **검색 범위** | 한정적 | 넓음 (구조적 + 의미적) |
| **정확도** | 질문 유형에 따라 변동 | 일관되게 높음 |
| **재현율(Recall)** | 제한적 | 높음 (중복 제거) |
| **적응성** | 특정 질문에 특화 | 다양한 질문 대응 |
| **복잡도** | 낮음 | 중간 (LangGraph로 관리) |
| **확장성** | 제한적 | 높음 (Agent 추가 용이) |

### 8.2 각 검색 방식의 강점 활용

**GraphRAG (Neo4j)가 효과적인 경우**:
- ✅ "경제 카테고리의 뉴스"
- ✅ "뉴스1에서 발행한 기사"
- ✅ "김철수 기자가 작성한 뉴스"
- ✅ 명확한 엔티티와 관계가 있는 질문

**VectorRAG (Elasticsearch)가 효과적인 경우**:
- ✅ "AI 관련 뉴스"
- ✅ "부산 박물관 소식"
- ✅ "해킹 사고 기사"
- ✅ 자연어 및 키워드 기반 질문

**Hybrid가 필요한 경우**:
- ✅ "경제 카테고리에서 네이버 관련 뉴스"
- ✅ "뉴스1의 AI 기사"
- ✅ 구조적 + 의미적 요소가 결합된 질문


## 9. LangGraph Supervisor 패턴의 장점


### 9.1 Supervisor 패턴의 핵심 가치

**1. 지능적 라우팅**
- Supervisor가 질문을 분석하고 최적의 Agent 선택
- 불필요한 검색 방지 (비용 및 시간 절약)

**2. 유연한 확장성**
```python
# 새로운 Agent 추가가 간단
workflow.add_node("news_summary_agent", news_summary_node)
workflow.add_node("sentiment_agent", sentiment_node)
```

**3. 명확한 제어 흐름**
- 상태 기반으로 Agent 간 정보 공유
- 조건부 라우팅으로 복잡한 워크플로우 구현

**4. 디버깅 용이**
- 각 노드의 입출력을 명확히 추적
- 워크플로우 시각화로 전체 흐름 파악

**5. Human-in-the-Loop**
```python
# 사람의 개입이 필요한 지점 추가 가능
workflow.add_node("human_review", human_review_node)
```


### 9.2 전통적인 Agent vs LangGraph Supervisor

| 측면 | 전통적 ReAct Agent | LangGraph Supervisor |
|------|-------------------|---------------------|
| **제어 흐름** | Tool 호출 순서가 불명확 | 명확한 그래프 구조 |
| **상태 관리** | 제한적 (메시지만) | 풍부한 전역 상태 |
| **Agent 조율** | LLM이 모든 결정 | Supervisor가 체계적 조율 |
| **확장성** | Tool 추가만 가능 | Agent 및 워크플로우 확장 |
| **병렬 실행** | 어려움 | 쉬움 (그래프 구조) |
| **디버깅** | 어려움 | 쉬움 (노드별 추적) |


## 10. 실전 응용 및 확장 아이디어


### 10.1 추가 Agent 아이디어

**1. 요약 Agent**
```python
def summary_agent_node(state):
    """검색된 뉴스를 요약"""
    # 긴 기사들을 짧게 요약
    pass
```

**2. 감성 분석 Agent**
```python
def sentiment_agent_node(state):
    """뉴스의 감성(긍정/부정/중립) 분석"""
    pass
```

**3. 트렌드 분석 Agent**
```python
def trend_agent_node(state):
    """시간대별 뉴스 트렌드 분석"""
    pass
```

**4. 추천 Agent**
```python
def recommendation_agent_node(state):
    """사용자 관심사 기반 뉴스 추천"""
    pass
```


### 10.2 실전 활용 시나리오

**1. 기업 내부 지식 검색 시스템**
- GraphRAG: 조직도, 프로젝트 관계
- VectorRAG: 문서, 이메일, 위키

**2. 고객 지원 시스템**
- GraphRAG: 제품 카탈로그, FAQ 관계
- VectorRAG: 고객 문의 이력, 매뉴얼

**3. 의료 정보 시스템**
- GraphRAG: 질병-증상-치료 관계
- VectorRAG: 의료 논문, 진료 기록

**4. 법률 검색 시스템**
- GraphRAG: 법조문 간 관계, 판례 인용
- VectorRAG: 법률 문서, 계약서


### 10.3 성능 개선 방안

**1. 캐싱 전략**
```python
from functools import lru_cache

@lru_cache(maxsize=100)
def cached_graph_search(query_type, param):
    """자주 검색되는 쿼리 캐싱"""
    return graph_rag_search(query_type, param)
```

**2. 병렬 검색**
```python
import asyncio

async def parallel_search(query):
    """GraphRAG와 VectorRAG를 병렬로 실행"""
    graph_task = asyncio.create_task(graph_search_async(query))
    vector_task = asyncio.create_task(vector_search_async(query))
    return await asyncio.gather(graph_task, vector_task)
```

**3. 결과 재순위화 (Re-ranking)**
```python
from sentence_transformers import CrossEncoder

def rerank_results(query, results):
    """Cross-Encoder로 결과 재순위화"""
    model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-12-v2')
    scores = model.predict([(query, doc) for doc in results])
    return sorted(zip(results, scores), key=lambda x: x[1], reverse=True)
```

**4. 적응형 검색 전략**
```python
def adaptive_search(query, history):
    """사용자 검색 히스토리 기반으로 전략 조정"""
    if user_prefers_structured(history):
        boost_graph = 1.5
    else:
        boost_graph = 1.0
    # ...
```


## 11. 리소스 정리


In [ ]:
# Neo4j 연결 종료
neo4j_conn.close()

# Elasticsearch 연결 종료 (자동)
print("✅ 모든 리소스 정리 완료")


## 12. 핵심 요약 및 학습 포인트
